# BronzeToSilver

## Visão Geral

Este notebook implementa um pipeline de transformação de dados da camada **Bronze** para a camada **Silver** utilizando o padrão **Medallion Architecture**, processando a tabela `tb_sales` com Spark Structured Streaming e gravação incremental via `MERGE` (upsert) em Delta.

## Fluxo do Notebook

1. **Configuração (Célula 2)**: Define os parâmetros do notebook via widgets:
   - `CATALOG`: nome do catálogo Unity Catalog
   - `CHECKPOINT_PATH`: localização do checkpoint do stream
   
   Com eles, monta as referências de origem (`{CATALOG}.bronze.tb_sales`) e destino (`{CATALOG}.silver.tb_sales`).

3. **Definição de Chaves e Colunas (Célula 4)**: Define:
   - `KEY_COLS`: colunas que compõem a chave natural de negócio (data da venda, marca/sabor, canal, embalagem, etc.)
   - `TEXT_COLS`: colunas textuais que recebem limpeza (trim)

4. **Transformação (Célula 5)**: A função `transform()` realiza a higienização e tipagem dos dados:
   - Conversão de `date` para `sale_date` (tipo date)
   - Casts seguros com `try_cast` para int, decimal e limpeza de espaços (`trim`)
   - Preserva as colunas de auditoria `_source_file` e `_ingestion_timestamp`
   - Gera uma `sales_key` (hash SHA-256 da chave natural composta)
   - Deduplica por `sales_key`, mantendo o registro com `_ingestion_timestamp` mais recente (exigência do MERGE: 1 linha de origem por chave)

5. **Ingestão Streaming com Upsert (Célula 6)**: A função `upsert_batch()` é executada via `foreachBatch`:
   - Se a tabela destino não existir, cria com `saveAsTable`
   - Caso exista, executa `MERGE` no Delta Lake por `sales_key` — atualizando registros correspondentes (`whenMatchedUpdateAll`) e inserindo novos (`whenNotMatchedInsertAll`)
   - O stream roda com `trigger(availableNow=True)` — processa os dados disponíveis e encerra, ideal para cargas em lote agendadas

## Características

- **Idempotente**: o MERGE por chave garante que reprocessamentos não dupliquem dados
- **Tolerante a erros de tipagem**: `try_cast` evita falhas por dados sujos (valores nulos em vez de exceção)
- **Rastreável**: colunas de auditoria indicam arquivo de origem e momento de ingestão


In [0]:
from pyspark.sql import functions as F, Window


dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("CHECKPOINT_PATH", "")

CATALOG = dbutils.widgets.get("CATALOG")
CHECKPOINT_PATH = dbutils.widgets.get("CHECKPOINT_PATH")

SOURCE = f"{CATALOG}.bronze.tb_sales"
TARGET = f"{CATALOG}.silver.tb_sales"

In [0]:
KEY_COLS = ["sale_date", "ce_brand_flvr", "btlr_org_lvl_c_desc", "trade_chnl_desc", "pkg_cat", "tsr_pckg_nm"]
TEXT_COLS = ["brand_nm", "btlr_org_lvl_c_desc", "chnl_group", "trade_chnl_desc", "pkg_cat", "pkg_cat_desc", "tsr_pckg_nm"]


In [0]:
def transform(df):

    out = df.select(
        F.expr("to_date(try_to_timestamp(trim(`date`), 'yyyy-MM-dd'))").alias("sale_date"),
        F.expr("try_cast(trim(ce_brand_flvr) as int)").alias("ce_brand_flvr"),
        *[F.trim(c).alias(c) for c in TEXT_COLS],
        F.expr("try_cast(trim(sales_volume) as decimal(18,2))").alias("sales_volume"),
        F.expr("try_cast(trim(`year`) as int)").alias("year"),
        F.expr("try_cast(trim(`month`) as int)").alias("month"),
        F.expr("try_cast(trim(`period`) as int)").alias("period"), 
        "_source_file",
        "_ingestion_timestamp",
    )
    out = out.withColumn("sales_key", F.sha2(F.concat_ws(
        "||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in KEY_COLS]), 256))

    # MERGE exige 1 linha de origem por chave: fica a mais recente
    w = Window.partitionBy("sales_key").orderBy(F.col("_ingestion_timestamp").desc())
    return out.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [0]:
def upsert_batch(batch_df, batch_id):
    from delta.tables import DeltaTable
    df = transform(batch_df)
    if not spark.catalog.tableExists(TARGET):
        df.write.format("delta").saveAsTable(TARGET)
        return
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(df.alias("s"), "t.sales_key = s.sales_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


(spark.readStream.table(SOURCE)
      .writeStream.foreachBatch(upsert_batch)
      .option("checkpointLocation", CHECKPOINT_PATH)
      .trigger(availableNow=True)
      .start()
      .awaitTermination())